# 마리음 ML: 녹음 형식(PCM16 · 16kHz · 모노) + 모델 전사 테스트 (Colab)

앱이 보낼 오디오 형식 그대로 녹음해서, ML 서버와 같은 모델·같은 설정으로 텍스트를 뽑아 보는 노트북이다.

| 단계 | 내용 |
|---|---|
| 1 | 브라우저 마이크로 **PCM16, 16kHz, 모노** 녹음 (앱 `record` 패키지 설정과 같은 형식) |
| 2 | 형식 확인: 바이트 수, 길이, 음량, 재생, 100ms(3,200바이트) 프레임 개수 |
| 3 | 모델 로드: `yoona-J/ASR_Whisper_Disease_General`, fp16 |
| 4 | 최종 전사: 서버와 같은 VAD 자르기 + 무음 판정 (`no_speech_prob > 0.6`) |
| 5 | 실시간 흉내: 0.5초마다 다시 전사 → `stable_text` / `unstable_text` (서버 interim 규칙) |
| 6 | 무음 환각 확인 |

- **런타임 → 런타임 유형 변경 → GPU(T4)** 로 바꾼 뒤 위에서부터 차례로 실행한다
- 녹음 셀을 실행하면 브라우저가 마이크 권한을 물어본다. 허용해야 한다
- GPU 서버(`10.1.2.11`)는 방화벽 때문에 Colab에서 접속할 수 없다. 그래서 이 노트북은 **같은 모델을 Colab GPU에서 직접** 돌린다
- 서버 코드와 차이: VAD는 Silero `get_speech_timestamps`로 근사했고, 실시간 흉내는 녹음이 끝난 뒤 시간 순서대로 재생해 보는 방식이다 (진짜 스트리밍 아님)

## 0. 설치와 GPU 확인

In [ ]:
!pip -q install transformers==4.53.3 silero-vad==6.2.3
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")

## 1. 마이크로 녹음 (PCM16 · 16kHz · 모노)

앱과 같은 형식을 만들기 위해 브라우저에서 바로 16kHz 모노로 받아 16-bit 정수(little-endian)로 바꾼다.
잡음 제거·자동 음량·에코 제거는 앱 설정과 같이 **끈다**.

`RECORD_SECONDS`를 바꿔 녹음 길이를 정한다 (서버 `utterance` 모드 상한은 10초).

In [ ]:
import base64
from IPython.display import Javascript, display
from google.colab.output import eval_js

RECORD_SECONDS = 5  # 1–10초

display(Javascript(r"""
async function recordPcm16(sec) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: {
    channelCount: 1, echoCancellation: false, noiseSuppression: false, autoGainControl: false}});
  const ctx = new AudioContext({sampleRate: 16000});
  const src = ctx.createMediaStreamSource(stream);
  const proc = ctx.createScriptProcessor(4096, 1, 1);
  const chunks = [];
  proc.onaudioprocess = e => chunks.push(new Float32Array(e.inputBuffer.getChannelData(0)));
  src.connect(proc); proc.connect(ctx.destination);
  const label = document.createElement('div');
  label.style = 'font-size:20px;color:#c00;margin:8px 0';
  document.body.appendChild(label);
  for (let s = sec; s > 0; s--) { label.textContent = '🎤 녹음 중… ' + s + '초'; await new Promise(r => setTimeout(r, 1000)); }
  label.textContent = '✅ 녹음 끝';
  proc.disconnect(); src.disconnect(); stream.getTracks().forEach(t => t.stop());
  const sr = ctx.sampleRate; await ctx.close();
  const n = chunks.reduce((a, c) => a + c.length, 0);
  const pcm = new Int16Array(n); let o = 0;
  for (const c of chunks) for (let i = 0; i < c.length; i++) {
    const s = Math.max(-1, Math.min(1, c[i])); pcm[o++] = s < 0 ? s * 32768 : s * 32767; }
  const bytes = new Uint8Array(pcm.buffer); let bin = '';
  for (let i = 0; i < bytes.length; i += 0x8000) bin += String.fromCharCode.apply(null, bytes.subarray(i, i + 0x8000));
  return {sr: sr, b64: btoa(bin)};
}
"""))
res = eval_js(f"recordPcm16({RECORD_SECONDS})")
pcm_bytes = base64.b64decode(res["b64"])   # ← 앱이 백엔드로 보내는 바이트와 같은 형식
SAMPLE_RATE = int(res["sr"])
print(f"받은 바이트: {len(pcm_bytes):,} / 브라우저 실제 샘플레이트: {SAMPLE_RATE} Hz")
if SAMPLE_RATE != 16000:
    print("⚠️ 브라우저가 16kHz를 지원하지 않아 다른 값으로 녹음됐다. 아래에서 16kHz로 변환한다 (ML 서버도 8–48kHz는 받아서 변환함)")

### (대안) 파일로 테스트하기
마이크 대신 휴대폰으로 녹음한 파일(m4a, wav, webm 등)을 올려서 같은 형식으로 바꿔 볼 수 있다.
이 셀은 **백엔드가 변환을 맡는 경우**(`ffmpeg -f s16le -ar 16000 -ac 1`)를 흉내 낸다. 필요할 때만 실행한다.

In [ ]:
from google.colab import files
import subprocess, os

up = files.upload()
name = next(iter(up))
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", name,
                "-f", "s16le", "-ar", "16000", "-ac", "1", "upload.pcm"], check=True)
pcm_bytes = open("upload.pcm", "rb").read()
SAMPLE_RATE = 16000
print(f"{name} → PCM16 16kHz 모노 {len(pcm_bytes):,} bytes")

## 2. 형식 확인 (ML 서버가 받는 방식 그대로 읽기)

In [ ]:
import numpy as np, wave
from IPython.display import Audio

FRAME_BYTES = 16000 * 2 // 10  # 100ms @ 16kHz PCM16 모노 = 3,200바이트
SR = 16000

assert len(pcm_bytes) % 2 == 0, "PCM16이면 바이트 수가 짝수여야 한다"
x = np.frombuffer(pcm_bytes, dtype="<i2").astype(np.float32) / 32768.0  # 서버 decode()와 같음
if SAMPLE_RATE != SR:
    import torchaudio
    x = torchaudio.functional.resample(torch.from_numpy(x), SAMPLE_RATE, SR).numpy()
    pcm_bytes = (np.clip(x, -1, 1) * 32767).astype("<i2").tobytes()

dur = len(x) / SR
peak = float(np.abs(x).max()) if len(x) else 0.0
rms_db = 20 * np.log10(np.sqrt((x ** 2).mean()) + 1e-10)
n_frames = int(np.ceil(len(pcm_bytes) / FRAME_BYTES))
print(f"길이 {dur:.2f}초 | 샘플 {len(x):,}개 | 바이트 {len(pcm_bytes):,} (= 초당 32,000 × {dur:.2f})")
print(f"최대 진폭 {peak:.3f} | 평균 음량 {rms_db:.1f} dBFS | 100ms 프레임 {n_frames}개")
if peak < 0.01: print("⚠️ 거의 무음이다. 마이크가 제대로 잡혔는지 확인")
if peak > 0.99: print("⚠️ 소리가 찢어졌다(클리핑). 마이크에서 조금 떨어져서 다시 녹음")
if dur > 10: print("⚠️ 10초 초과: 실제 서버(utterance)에서는 AUDIO_TOO_LONG")

with wave.open("recorded_16k_mono.wav", "wb") as w:   # 들어 보고 내려받을 수 있게 저장
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(pcm_bytes)
display(Audio(x, rate=SR))
print("속도·음높이가 정상으로 들리면 형식이 맞다 (빠르거나 낮으면 샘플레이트/채널이 어긋난 것)")

## 3. 모델 로드 (ML 서버와 같은 모델, fp16)

In [ ]:
import time
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.modeling_outputs import BaseModelOutput
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

MODEL = "yoona-J/ASR_Whisper_Disease_General"
dev, dtype = torch.device("cuda"), torch.float16
t = time.time()
processor = WhisperProcessor.from_pretrained(MODEL)
model = WhisperForConditionalGeneration.from_pretrained(MODEL, torch_dtype=dtype).to(dev).eval()
tok = processor.tokenizer
FDI = processor.get_decoder_prompt_ids(language="ko", task="transcribe")  # 노트북·서버와 같음 (config의 <|en|>을 덮어씀)
SPECIAL, EOT = set(tok.all_special_ids), tok.eos_token_id
SOT = tok.convert_tokens_to_ids("<|startoftranscript|>")
NO_SPEECH_ID = next(i for i in (tok.convert_tokens_to_ids(s) for s in ("<|nospeech|>", "<|nocaptions|>"))
                    if i is not None and i != tok.unk_token_id)
print(f"로드 {time.time() - t:.1f}초")

@torch.inference_mode()
def transcribe(audio16k, max_new_tokens=128):
    """server/engine.py와 같은 계산: 인코더 1회 → no_speech_prob + greedy generate → confidence"""
    t0 = time.perf_counter()
    feats = processor([audio16k], sampling_rate=SR, return_tensors="pt").input_features.to(dev, dtype)
    enc = model.model.encoder(feats)
    sot = torch.full((1, 1), SOT, device=dev)
    dec = model.model.decoder(input_ids=sot, encoder_hidden_states=enc.last_hidden_state)
    no_speech = torch.softmax(model.proj_out(dec.last_hidden_state[:, -1]).float(), -1)[0, NO_SPEECH_ID].item()
    out = model.generate(encoder_outputs=BaseModelOutput(last_hidden_state=enc.last_hidden_state),
                         forced_decoder_ids=FDI, return_dict_in_generate=True, output_scores=True,
                         max_new_tokens=max_new_tokens)
    torch.cuda.synchronize()
    T = len(out.scores)
    gen = out.sequences[0, -T:].tolist()
    lp = []
    for i, tid in enumerate(gen):
        if tid == EOT: break
        if tid not in SPECIAL:
            lp.append(torch.log_softmax(out.scores[i][0].float(), -1)[tid].item())
    return {"text": processor.batch_decode(out.sequences, skip_special_tokens=True)[0].strip(),
            "confidence": round(float(np.mean(np.exp(lp))), 4) if lp else 0.0,
            "no_speech_prob": round(no_speech, 4),
            "gpu_ms": round((time.perf_counter() - t0) * 1000)}

transcribe(np.zeros(SR, np.float32), 8)  # 워밍업 (서버도 시작할 때 한 번 돌림)
print("준비 완료")

## 4. 최종 전사 (서버 `transcript_final`과 같은 규칙)

1. Silero VAD로 말소리를 찾는다 (threshold 0.5, 최소 말소리 150ms)
2. 첫 말소리 −200ms ~ 마지막 말소리 +200ms만 남긴다. 문장 중간 쉼은 그대로 둔다
3. 말소리가 없으면 모델을 돌리지 않고 빈 전사
4. 모델 결과의 `no_speech_prob > 0.6`이면 빈 전사 (이 모델은 무음에서 문장을 지어내기 때문)

In [ ]:
import json
from silero_vad import load_silero_vad, get_speech_timestamps

vad_model = load_silero_vad()
NO_SPEECH_THRESHOLD = 0.6
PAD = int(0.2 * SR)

def speech_spans(audio):
    vad_model.reset_states()
    return get_speech_timestamps(torch.from_numpy(audio), vad_model, sampling_rate=SR,
                                 threshold=0.5, min_speech_duration_ms=150, speech_pad_ms=0)

def final_message(audio, trace_id="colab-test"):
    spans = speech_spans(audio)
    if not spans:  # 말소리 없음 → Whisper 생략
        text, conf, nsp, info = "", 0.0, 1.0, "VAD: 말소리 없음 → 모델 생략"
    else:
        a, b = max(0, spans[0]["start"] - PAD), min(len(audio), spans[-1]["end"] + PAD)
        r = transcribe(audio[a:b])
        silenced = r["no_speech_prob"] > NO_SPEECH_THRESHOLD
        text = "" if silenced else r["text"]
        conf, nsp = (r["confidence"] if text else 0.0), r["no_speech_prob"]
        info = (f"VAD 구간 {a / SR:.2f}–{b / SR:.2f}초 (말소리 {len(spans)}덩어리) | GPU {r['gpu_ms']}ms"
                + (f" | 무음 판정으로 지움 (모델 출력: {r['text']!r})" if silenced else ""))
    print(info)
    return {"type": "transcript_final", "trace_id": trace_id, "text": text, "confidence": conf,
            "duration_ms": int(len(audio) / SR * 1000), "model_version": MODEL,
            "no_speech_prob": nsp, "segment_id": 0, "is_last": True}

print(json.dumps(final_message(x), ensure_ascii=False, indent=1))

## 5. 실시간 흉내 (서버 `transcript_interim` 규칙)

녹음한 소리를 100ms 프레임으로 다시 흘려 보내면서 서버와 같은 규칙으로 중간 전사를 만든다.

- 말소리가 300ms 이상 쌓인 뒤부터, **새 말소리가 있을 때만 0.5초마다** 지금까지의 음성 전체를 다시 전사 (`text`는 누적)
- **LocalAgreement-2:** 연속한 두 결과가 같은 앞 단어만 `stable_text`로 확정, 마지막 단어는 확정하지 않음
- 녹음이 끝나면(`end`) 최종 전사 1개 (`is_last: true`)

진짜 스트리밍이 아니라서 지연 시간은 GPU 처리 시간만 보여 준다.

In [ ]:
import re
_PUNCT = re.compile(r"[.,?!~·…]")
_norm = lambda w: _PUNCT.sub("", w)

class Agreement:  # server/stream.py와 같은 규칙
    def __init__(self):
        self.committed, self.prev = [], None
    def update(self, text):
        words = text.split()
        n = len(self.committed)
        consistent = [_norm(w) for w in words[:n]] == [_norm(w) for w in self.committed]
        if self.prev is not None and consistent:
            lcp = 0
            for a, b in zip(self.prev, words):
                if _norm(a) != _norm(b): break
                lcp += 1
            lcp = min(lcp, len(words) - 1)
            if lcp > n: self.committed = words[:lcp]
        self.prev = words
        n = len(self.committed)
        return " ".join(self.committed), " ".join(words[n:])

FRAME = SR // 10            # 100ms
STEP, MIN_SPEECH = 0.5, 0.3  # 초

spans = speech_spans(x)
def speech_until(t):  # t초까지 쌓인 말소리 길이(초)
    return sum(max(0, min(s["end"], t * SR) - s["start"]) for s in spans) / SR

agree, last_t, last_speech = Agreement(), 0.0, 0.0
print(f"{'시각':>6}  stable(진하게) | unstable(흐리게)")
for k in range(1, len(x) // FRAME + 1):
    t = k * FRAME / SR
    sp = speech_until(t)
    if sp < MIN_SPEECH or t - last_t < STEP or sp <= last_speech:
        continue
    a = max(0, spans[0]["start"] - PAD)
    r = transcribe(x[a:k * FRAME], max_new_tokens=96)
    stable, unstable = agree.update(r["text"])
    last_t, last_speech = t, sp
    msg = {"type": "transcript_interim", "text": r["text"], "stable_text": stable,
           "unstable_text": unstable, "confidence": r["confidence"], "is_final": False,
           "segment_id": 0, "audio_end_ms": int(t * 1000)}
    print(f"{t:5.1f}s  [{stable}] | {unstable}    (GPU {r['gpu_ms']}ms)")
    print("        ", json.dumps(msg, ensure_ascii=False))

print("\n── end ──")
print(json.dumps(final_message(x), ensure_ascii=False))

## 6. 무음 환각 확인

이 모델은 소리가 없어도 문장을 지어낸다. 서버는 VAD와 `no_speech_prob`로 막는다.

In [ ]:
silence = np.zeros(SR, np.float32)  # 무음 1초
noise = (np.random.default_rng(0).standard_normal(SR) * 0.01).astype(np.float32)  # 약한 백색잡음 1초

for name, a in [("무음 1초", silence), ("백색잡음 1초", noise)]:
    raw = transcribe(a)
    print(f"[{name}] 모델 그대로: {raw['text']!r} (confidence {raw['confidence']}, no_speech {raw['no_speech_prob']})")
    print(f"[{name}] 서버 규칙 적용: {final_message(a)['text']!r}\n")

## 7. 결과 파일 내려받기 (선택)

- `recorded_16k_mono.wav`: 녹음한 소리 (16kHz 모노 PCM16 WAV). 팀 공유나 서버 시험(`scripts/stream_client.py --wav`)에 쓸 수 있다

In [ ]:
from google.colab import files
files.download("recorded_16k_mono.wav")